# Input

- 4 csv files

## Data Understanding

I'm not an insurance person, so before writing any code I worked through
what each file actually represents in plain terms.

**The big picture:** it's like a sales CRM pipeline, but for insurance.
Every week, for each of 8 business lines, four things get tracked:

Did we get asked to quote business, and what happened to it? (submissions file)

How much money did we actually collect vs. what we expected? (premium file)

What deals are still "in progress," not yet decided? (pipeline file)

How many claims came in, and how much are they costing us? (loss file)

All four files join on the same two columns:
`week_ending` + `lob` (line of business) — no other keys needed.

### `case4_weekly_submissions.csv` — what happened to each deal

Every new risk a broker brings us ends up in exactly one bucket:

| Column | Meaning |
|---|---|
| `week_ending` | The Sunday closing that week |
| `lob` | Business line (Cyber, Environmental, etc.) |
| `submissions_count` | New deals that came in this week |
| `quoted_count` | We gave a price, still pending the customer's decision |
| `bound_count` | Became an actual signed policy (a win) |
| `declined_count` | We said no outright |
| `ntu_count` | "Not Taken Up" — we quoted, customer went elsewhere (lost the bid, not our refusal) |

**Hit rate** = `bound / (bound + quoted + declined + ntu)` — a conversion
rate, same idea as a sales funnel win %.

### `case4_weekly_premium.csv` — the money

| Column | Meaning |
|---|---|
| `actual_gwp` | Actual premium revenue booked this week for this line ("GWP" = Gross Written Premium) |
| `plan_gwp` | Budgeted/target revenue for this week |
| `ytd_actual` / `ytd_plan` | Running totals of the above since the start of the year |

The whole story here is just actual vs. plan — same as "are we ahead or
behind our quarterly sales target."

### `case4_pipeline.csv` — deals still in limbo

| Column | Meaning |
|---|---|
| `open_quotes_count` | Quotes currently pending a customer decision |
| `open_quotes_gwp_est` | Estimated £ value of those pending quotes if they all closed |
| `avg_days_in_pipeline` | Average time those open quotes have been stuck unresolved |

**Note:** I checked this file closely for a hidden signal and it's
mostly noise — no clean trend for any business line across the 12
weeks. Worth knowing that's a deliberate finding, not a gap in my
analysis.

### `case4_loss_indicators.csv` — what claims are costing us

| Column | Meaning |
|---|---|
| `new_claims_count` | New claims reported this week |
| `new_claims_incurred_est` | Estimated £ cost of those new claims |
| `attritional_loss_ratio_ytd` | Cumulative (claims paid ÷ premium collected) since the start of the year, as a fraction (e.g. `0.554` = 55.4%) |

**Loss ratio** is the headline profitability metric: under ~60% is
healthy, climbing past it means the line is paying out more in claims
relative to premium than it should. Important quirk: this column is
already a running year-to-date figure, not a "this week only" number —
week 12 already has weeks 1–11 baked in, so it plots as a smooth trend
without needing my own rolling calculation. "Attritional" just means
ordinary, expected claims (as opposed to one catastrophic loss) — not
relevant to how I use the column.

# Output

- Sample executive narrative output
- Dashboard
- Prompt file(s)

# Thoughts

## 1. build the project structure

built as in Implementation

## 2. Signal detector

I have 4 confirmed signals, each needing a rule that requires a sustained pattern — never trigger off a single week. The detector lives in `detect.py`, completely separate from the LLM. Ranking is deterministic in code; the LLM only writes the narrative from results I've already decided.

Each signal is one function returning a dict (lob, type, title, detail, action) or None. A `detect_all()` wrapper calls all four and returns the non-None results. All thresholds live in `config.py` so they can be tuned without touching logic.

- **Signal 1 — Excess Casualty:** actual GWP(revenue) below 60% of plan in at least 10 of 12 weeks — structural, not a blip. → premium file only
- **Signal 2 — Cyber hit rate:** average hit rate weeks 1–8 vs weeks 9–12 drops by more than 10pp — sustained collapse, not a one-off. submissions file only
- **Signal 3 — Environmental loss ratio:** latest YTD loss ratio exceeds week-1 value by more than 10pp *and* is above the 60% target — direction matters as much as level. → loss file only
- **Signal 4 — Political Violence:** GWP beats plan in at least 8 of 12 weeks *and* loss ratio stays below 60% — growth is only an opportunity if it's profitable. → premium and loss files combined
## 3. Prompt files
## 4. Agent orchestrator
## 5. Dashboard

# Implementation

read the data

In [1]:
import sys

from src.agent import save_json

sys.path.insert(0, "src")

from agent import read_data

loss, pipeline, premium, submissions = read_data()

In [2]:
print(premium.head())
print(premium.dtypes)

  week_ending                      lob  actual_gwp  plan_gwp  ytd_actual  \
0  2024-07-07                    Cyber    188100.0    180000    188100.0   
1  2024-07-07  Transactional Liability    202500.0    220000    202500.0   
2  2024-07-07            Environmental     80300.0     80000     80300.0   
3  2024-07-07           Political Risk    101900.0     95000    101900.0   
4  2024-07-07       Political Violence     59900.0     45000     59900.0   

   ytd_plan  
0    180000  
1    220000  
2     80000  
3     95000  
4     45000  
week_ending    datetime64[us]
lob                       str
actual_gwp            float64
plan_gwp                int64
ytd_actual            float64
ytd_plan                int64
dtype: object


# signal detectors

In [3]:
from detect import detect_gwp_underperformance

signal1 = detect_gwp_underperformance(premium)

In [4]:
print(signal1)

[{'lob': 'Excess Casualty', 'rank': np.int64(9), 'type': 'concern', 'title': 'Excess Casualty: Structural GWP underperformance', 'detail': 'Actual GWP was below 60% of plan in 9 of 12 weeks.', 'action': 'Check what is underperforming and adjust accordingly'}]


In [5]:
from detect import detect_hit_rate_collapse

signal2 = detect_hit_rate_collapse(submissions)

In [6]:
print(signal2)

[{'lob': 'Cyber', 'rank': np.float64(0.17995136379003657), 'type': 'concern', 'title': 'Hit rate collapse in weeks 9–12', 'detail': 'Hit rate averaged 27% in weeks 1–8, dropping to 9% in weeks 9–12.', 'action': 'Review pricing against market — rising NTU suggests competitors are offering cheaper terms.'}]


In [7]:
from detect import detect_loss_ratio_trend

signal3 = detect_loss_ratio_trend(loss)

In [8]:
print(signal3)

[{'lob': 'Environmental', 'rank': np.float64(0.19099999999999995), 'type': 'concern', 'title': 'Loss ratio deteriorating above target', 'detail': 'Attritional loss ratio has risen from 55% at week 1 to 74% at week 12 — 19% above starting point and beyond the 60% target.', 'action': 'Flag for claims review and assess whether recent bound business warrants a rate adequacy check.'}]


In [9]:
from detect import detect_gwp_outperformance

signal4 = detect_gwp_outperformance(premium, loss)

In [10]:
print(signal4)

[{'lob': 'Political Risk', 'rank': np.int64(9), 'type': 'opportunity', 'title': 'Consistent outperformance with healthy loss ratio', 'detail': 'GWP exceeded plan in 9 of 12 weeks. Loss ratio stands at 44% — well below the 60% target. Growth is profitable.', 'action': 'Assess capacity headroom. If the book can absorb more volume at current margins, consider increasing the plan allocation.'}, {'lob': 'Political Violence', 'rank': np.int64(12), 'type': 'opportunity', 'title': 'Consistent outperformance with healthy loss ratio', 'detail': 'GWP exceeded plan in 12 of 12 weeks. Loss ratio stands at 32% — well below the 60% target. Growth is profitable.', 'action': 'Assess capacity headroom. If the book can absorb more volume at current margins, consider increasing the plan allocation.'}]


In [11]:
from detect import detect_all

print(detect_all(premium, submissions, loss))

[{'lob': 'Political Violence', 'rank': np.int64(12), 'type': 'opportunity', 'title': 'Consistent outperformance with healthy loss ratio', 'detail': 'GWP exceeded plan in 12 of 12 weeks. Loss ratio stands at 32% — well below the 60% target. Growth is profitable.', 'action': 'Assess capacity headroom. If the book can absorb more volume at current margins, consider increasing the plan allocation.'}, {'lob': 'Excess Casualty', 'rank': np.int64(9), 'type': 'concern', 'title': 'Excess Casualty: Structural GWP underperformance', 'detail': 'Actual GWP was below 60% of plan in 9 of 12 weeks.', 'action': 'Check what is underperforming and adjust accordingly'}, {'lob': 'Political Risk', 'rank': np.int64(9), 'type': 'opportunity', 'title': 'Consistent outperformance with healthy loss ratio', 'detail': 'GWP exceeded plan in 9 of 12 weeks. Loss ratio stands at 44% — well below the 60% target. Growth is profitable.', 'action': 'Assess capacity headroom. If the book can absorb more volume at current m

## prompt generation

In [12]:
from agent import build_prompt
from detect import detect_all

signals = detect_all(premium, submissions, loss)
week_ending = premium["week_ending"].max().strftime("%Y-%m-%d")

system_prompt, user_prompt = build_prompt(week_ending, signals, premium)
print(system_prompt)
print(user_prompt)

You are an analyst working for the Cheif Underwriting Officer (CUO) of Mosaic Insurance. Your job is helping the CUO
to analyze the data and generate reports.

Rules:
1. Be accurate with the data, the original data should never be modified during the process. You may only use the numbers
given to you, never assume or estimate anything. If something is missing, you can ask the user for it.
2. Be precise, dont give me 10 pages of texts which is hard to read.
3. Be action oriented, I need to proceed some action, give me the reason, and proof(maybe mainly from data).
4. Also this is writing for a CUO, not data scientist. So make sure the report is easy to understand.

Output Format:
1. One sentence summary
2. Tell me the concerns, why, and what I need to do.
3. Tell me the opportunities, why, and what I can do.
Report for week ending on 2024-09-22

Detected signals:

        Title: Consistent outperformance with healthy loss ratio
        Detail: GWP exceeded plan in 12 of 12 weeks. Loss r

## API call test

In [13]:
from agent import call_claude
narrative = call_claude(system_prompt, user_prompt)

print(narrative)

# Weekly Underwriting Report
**Week Ending 2024-09-22**

---

## Summary
Portfolio is broadly on track with several lines outperforming plan, but Excess Casualty is a material drag, and two lines show emerging risk signals requiring immediate attention.

---

## Concerns

**1. Excess Casualty — Structural Underperformance**
GWP was below 60% of plan in 9 of 12 weeks, resulting in a YTD attainment of only 58%. This is not a short-term blip — it is a persistent pattern. **Action needed:** Identify the root cause (pipeline, pricing, appetite, or capacity) and either reset the plan or intervene commercially.

**2. Loss Ratio Deterioration (unnamed line)**
Attritional loss ratio has climbed from 55% at Week 1 to 74% at Week 12 — a 19-point deterioration, now 14 points above the 60% target. **Action needed:** Trigger a claims review immediately and run a rate adequacy check on business bound in recent weeks.

**3. Hit Rate Collapse (unnamed line)**
Hit rate fell from an average of 27% in Wee

Generate dashboard

In [14]:
from agent import generate_dashboard
output = save_json(narrative, signals, premium, submissions)
generate_dashboard(output)  # output is the dict returned by save_json()